# Batch Monthly Statistics & Feature Engineering

**Project**: FLAPS  
**Purpose**: Compute monthly aggregated weather statistics and engineer features for all CSV files in a specified folder

---

## Table of Contents
1. [Setup](#1-setup)
2. [Configuration](#2-configuration)
3. [Load and Process All Files](#3-load-and-process-all-files)
4. [Compute Features](#4-compute-features)
5. [Export Results](#5-export-results)

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import glob
import os
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

## 2. Configuration

Set the input folder path and file extension to process all weather CSV files in one folder (i.e. one city). The city names are in their three-letter abbreviation form.

In [3]:
# [USER INPUT] Define the city to process
city = "mel"

# Append city name to relevant folder and file names
input_folder = "../data/raw/"+city
file_pattern = "*.csv"
output_file  = "../data/processed/features_"+city+".csv"

# List all matching files sorted by date
files = sorted(glob.glob(os.path.join(input_folder,file_pattern)))

print(f"Found {len(files)} CSV files to process in this folder.")
print(f"\nFirst 5 files:")
for f in files[:5]:
    print(f"  - {os.path.basename(f)}")

Found 205 CSV files to process in this folder.

First 5 files:
  - melbourne_airport-200901.csv
  - melbourne_airport-200902.csv
  - melbourne_airport-200903.csv
  - melbourne_airport-200904.csv
  - melbourne_airport-200905.csv


## 3. Load and Process All Files

Load all CSV files and combine them into a single DataFrame. Each file follows the same format from the Bureau of Meteorology.

In [4]:
# Function to load and clean a single CSV file. See notebook data_cleaning_single.ipynb for further details.
def load_clean_csv(file_path):
    """Load and clean a single weather CSV file."""
    # Load CSV and remove unnecessary rows
    df = pd.read_csv(file_path, skiprows=range(1,13), skipfooter=1, 
                    encoding='latin-1', engine='python')
    
    # Rename columns
    df.columns = ['station', 'date', 'evapotranspiration', 'rain', 'panevaporation', 
                    'temp_max', 'temp_min', 'hum_max', 'hum_min', 'wind_speed', 'solar_rad']
    
    # Drop station column since it is the same for all entries
    df = df.drop('station', axis=1)
    
    # Convert date to datetime for easier sorting
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    
    # Convert numeric columns
    numeric_cols = ['evapotranspiration', 'rain', 'panevaporation', 
                    'temp_max', 'temp_min', 'hum_max', 'hum_min', 'wind_speed', 'solar_rad']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Extract year and month for grouping. Expecially important to compute monthly stats.
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['year_month'] = df['date'].dt.to_period('M')
    
    return df


Loop and run load_clean_csv() for every csv files in the folder.

In [5]:
# Create temporary variable
df_temp = []

for i, f in enumerate(files,1):
    df_temp.append(load_clean_csv(f))
    if i%50 == 0:
        print(f"Processed {i}/{len(files)} files...")

# df_all is a list, need to concatanate into 1 dataframe.
df_combined = pd.concat(df_temp, ignore_index=True)

print(f"\n{'='*80}")
print(f"Data cleaning done!")
print(f"{'='*80}")
print(f"Total files processed: {len(df_temp)}")
print(f"Total records: {len(df_combined):,} days")
print(f"Date range: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")

Processed 50/205 files...
Processed 100/205 files...
Processed 150/205 files...
Processed 200/205 files...

Data cleaning done!
Total files processed: 205
Total records: 6,212 days
Date range: 2009-01-01 to 2026-01-08


## 4. Compute Features

Compute all monthly statistics and engineered features following the procedure described in notebook *compute_monthly_stats_feature_batch.ipynb*.

In [6]:
print("Computing basic monthly statistics...")

# Core monthly statistics
monthly_stats = df_combined.groupby('year_month').agg({
    # Number of days in the month
    'date': 'count',      

    # Temperature features
    'temp_max': ['max', 'mean'],
    'temp_min': ['min', 'mean'],
    
    # Rainfall features
    'rain': ['max', 'mean'],
    
    # Wind features
    'wind_speed': ['mean', 'max'],
    
    # Humidity features
    'hum_max': 'mean',
    'hum_min': 'mean'
}).reset_index()

# Flatten column names
columns_combined = []
for col in monthly_stats.columns.values:
    if col[1] == "":
        columns_combined.append(col[0])
    else:
        columns_combined.append("_".join(col))

monthly_stats.columns = columns_combined

# Rename aggregate columns to be more descriptive
monthly_stats.rename(columns={
    'date_count': 'days_in_month',
    'rain_mean': 'avg_rainfall_per_day',
    'temp_max_max': 'max_temperature',
    'temp_max_mean': 'avg_max_temp',
    'temp_min_min': 'min_temperature',
    'temp_min_mean': 'avg_min_temp',
    'rain_max': 'max_daily_rainfall',
    'wind_speed_mean': 'avg_wind_speed',
    'wind_speed_max': 'max_wind_speed',
    'hum_max_mean': 'avg_max_humidity',
    'hum_min_mean': 'avg_min_humidity'
}, inplace=True)

print(f"  ✓ Basic statistics computed ({len(monthly_stats)} months)")

Computing basic monthly statistics...
  ✓ Basic statistics computed (205 months)


In [7]:
print("Computing temperature features...")

# Temperature feature engineering
temp_features = df_combined.groupby('year_month').apply(lambda x: pd.Series({
    # Temperature range (daily variability)
    'temp_range_mean': (x['temp_max'] - x['temp_min']).mean(),
    
    # Extreme temperature days
    'days_above_35C': (x['temp_max'] > 35).sum(),
    
    # Temperature volatility (day-to-day changes)
    'temp_volatility': x['temp_max'].diff().abs().mean()
}), include_groups=False).reset_index()

# Merge with monthly stats
monthly_stats = monthly_stats.merge(temp_features, on='year_month', how='left')

print(f"  ✓ Temperature features added")

Computing temperature features...
  ✓ Temperature features added


In [8]:
print("Computing precipitation features...")

# Precipitation feature engineering
precip_features = df_combined.groupby('year_month').apply(lambda x: pd.Series({
    # Rainy days
    'rainy_days': (x['rain'] > 0).sum(),
    'heavy_rain_days': (x['rain'] > 10).sum(),  # >10mm considered heavy
    
    # Rain intensity on rainy days
    'avg_rainfall_on_rainy_days': x[x['rain'] > 0]['rain'].mean() if (x['rain'] > 0).any() else 0
}), include_groups=False).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(precip_features, on='year_month', how='left')

print(f"  ✓ Precipitation features added")

Computing precipitation features...
  ✓ Precipitation features added


In [9]:
print("Computing wind features...")

# Wind feature engineering
wind_features = df_combined.groupby('year_month').apply(lambda x: pd.Series({
    # High wind days (thresholds based on aviation standards)
    'days_high_wind': (x['wind_speed'] > 8).sum(),  # >8 m/s (~15 knots)
    
    # Wind variability
    'wind_speed_std': x['wind_speed'].std()
}), include_groups=False).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(wind_features, on='year_month', how='left')

print(f"  ✓ Wind features added")

Computing wind features...
  ✓ Wind features added


In [10]:
print("Computing humidity features...")

# Humidity feature engineering
humidity_features = df_combined.groupby('year_month').apply(lambda x: pd.Series({
    # High humidity days (potential for fog/low visibility)
    'days_high_humidity': (x['hum_max'] > 90).sum()
}), include_groups=False).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(humidity_features, on='year_month', how='left')

print(f"  ✓ Humidity features added")

Computing humidity features...
  ✓ Humidity features added


In [11]:
print("Computing extreme weather features...")

# Extreme weather event features
extreme_features = df_combined.groupby('year_month').apply(lambda x: pd.Series({
    # Extreme weather days
    'extreme_weather_days': ((x['temp_max'] > 35) | (x['rain'] > 10) | 
                            (x['wind_speed'] > 8) | (x['hum_max'] > 95)).sum()
}), include_groups=False).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(extreme_features, on='year_month', how='left')

print(f"  ✓ Extreme weather features added")

Computing extreme weather features...
  ✓ Extreme weather features added


In [12]:
# Display complete monthly statistics
print("="*80)
print("MONTHLY WEATHER STATISTICS SUMMARY")
print("="*80)
print(f"\nTotal months analyzed: {len(monthly_stats)}")
print(f"Total features created: {len(monthly_stats.columns) - 1}")  # -1 for year_month
print(f"\nFeature breakdown:")
print(f"  - Basic statistics: 9")
print("  - Temperature features: 3")
print("  - Precipitation features: 3")
print("  - Wind features: 2")
print("  - Humidity features: 1")
print("  - Extreme weather features: 1")
print(f"\nTotal: {len(monthly_stats.columns) - 1} features")

print("\n" + "="*80)
print("Sample monthly statistics (first 10 rows):")
print(monthly_stats.head(10))

print("\n" + "="*80)
print("Sample monthly statistics (last 10 rows):")
print(monthly_stats.tail())

MONTHLY WEATHER STATISTICS SUMMARY

Total months analyzed: 205
Total features created: 21

Feature breakdown:
  - Basic statistics: 9
  - Temperature features: 3
  - Precipitation features: 3
  - Wind features: 2
  - Humidity features: 1
  - Extreme weather features: 1

Total: 21 features

Sample monthly statistics (first 10 rows):
  year_month  days_in_month  max_temperature  avg_max_temp  min_temperature  \
0    2009-01             31             44.2         28.80              6.3   
1    2009-02             28             46.8         27.86              6.9   
2    2009-03             31             34.7         24.24              8.4   
3    2009-04             30             31.8         20.25              1.2   
4    2009-05             31             23.0         17.04              3.2   
5    2009-06             30             18.1         14.34              1.4   
6    2009-07             31             18.0         13.90              0.9   
7    2009-08             31       

## 5. Export Results

Save the computed monthly statistics for use in modeling.

In [13]:
# Ensure output directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Export to CSV
monthly_stats.to_csv(output_file, index=False)
print(f"Monthly statistics exported to: {output_file}")
print(f"Total records exported: {len(monthly_stats)}")

# Display column list for reference
print("\nAll features available for modeling:")
print("="*80)
for i, col in enumerate(monthly_stats.columns, 1):
    print(f"{i:2d}. {col}")

Monthly statistics exported to: ../data/processed/features_mel.csv
Total records exported: 205

All features available for modeling:
 1. year_month
 2. days_in_month
 3. max_temperature
 4. avg_max_temp
 5. min_temperature
 6. avg_min_temp
 7. max_daily_rainfall
 8. avg_rainfall_per_day
 9. avg_wind_speed
10. max_wind_speed
11. avg_max_humidity
12. avg_min_humidity
13. temp_range_mean
14. days_above_35C
15. temp_volatility
16. rainy_days
17. heavy_rain_days
18. avg_rainfall_on_rainy_days
19. days_high_wind
20. wind_speed_std
21. days_high_humidity
22. extreme_weather_days


In [14]:
# Verification - check for any missing values
print("\nData Quality Check:")
print("="*80)
print("Missing values per column:")
print(monthly_stats.isnull().sum())

print("\nBasic statistics:")
print(monthly_stats.describe())


Data Quality Check:
Missing values per column:
year_month                    0
days_in_month                 0
max_temperature               0
avg_max_temp                  0
min_temperature               0
avg_min_temp                  0
max_daily_rainfall            0
avg_rainfall_per_day          0
avg_wind_speed                0
max_wind_speed                0
avg_max_humidity              0
avg_min_humidity              0
temp_range_mean               0
days_above_35C                0
temp_volatility               0
rainy_days                    0
heavy_rain_days               0
avg_rainfall_on_rainy_days    0
days_high_wind                0
wind_speed_std                0
days_high_humidity            0
extreme_weather_days          0
dtype: int64

Basic statistics:
       days_in_month  max_temperature  avg_max_temp  min_temperature  \
count         205.00           205.00        205.00           205.00   
mean           30.30            29.34         20.63             4.39   
